# 04 · Develop your prompt, on dev only

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/04_develop.ipynb)

Assemble a prompt from moves you can name, and change it for reasons you can state.

```
  01_build_pool_<track>  →  02_sample  →  02b_add_samples  →  03_annotate  →▶ 04_develop  →  05_test  →  06_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_dev.json` (from 03) · the pool (from 01) |
| **Writes** | `outputs/<track>_<group>_rounds.json` · `..._round_notes.json` · your prompt files |

---

Everything from here on is measured against **your** gold set, not the corpus's labels. That is the point of the last three notebooks.

**This notebook never opens the held-out set.** That is why it is a separate file from `05_test.ipynb` rather than a section further down: a boundary you have to open another notebook to cross is one you cannot drift across while you are concentrating on something else.

Every number here is a **dev** number. You use them to decide what to change next. None of them is the number you report.

> **Free-tier pacing.** The backend waits a few seconds between calls and retries on rate-limit errors, so a run takes minutes and may print `(rate limited - waiting Ns then retrying)`. That is normal — and it is why you iterate on dev: a dozen or so items is about a minute per round, so you get enough rounds to actually learn something.

> **On the size of this study.** One call per item, four-and-a-bit seconds apart, no batching: forty items is minutes and four hundred is most of an afternoon of a quota you share with everyone else on the course. A study that could support a claim about a corpus needs hundreds of items per class. This one cannot, and that is a limitation to state in your report's limitations section rather than write around. What transfers is the method — the split, the freezing, the audit trail — not the number.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, DISAGREED_PATH,
                    ADJUDICATED_PATH, ROUNDS_PATH, NOTES_PATH, PROMPT_FILE,
                    SHEET_PATH, describe)

# Files in, files out, and the connection to the model: all plumbing.
from pipeline import (load_gold, label_set, load_prompt, save_prompt,
                      save_json, setup, build_fewshot)

# Scoring. `evaluate` prints per-class precision, recall and F1, Cohen's κ and
# the confusion matrix, and hands back the macro-F1 — the Day 2 S6 Part B call.
from metrics import evaluate

# `run_prompt`, `extract_label` and `show_errors` are NOT imported. They are the
# three that decide what the model said, what counts as a label and what counts
# as an error, so they are cells further down that you can read and change. Run
# those cells before the rounds that use them.
#
# These are what those three cells call. `_default_backend` is the connection
# `setup()` opened and `_one_prediction_line` is the one-line-per-item printout;
# both are plumbing, so they stay imported rather than filling a cell.
import re
import pandas as pd
from pipeline import _default_backend, _one_prediction_line

# `freeze_test_run` is NOT imported here — and neither is TEST_PATH, PRED_PATH
# or TESTLOG_PATH. They are left out of the config import above on purpose, so
# there is no name in this notebook that reaches the held-out set. Not a rule
# you have to remember: a NameError if you try. Those live in 05_test.ipynb.

describe()                  # what this notebook is working on


## Connect to the model

Now we open the connection this notebook will send every prompt through. It gets a cell of its own because it does something the plumbing cell above does not: it reaches out to a service, and what it prints back is worth reading.

**The three settings are handed over here in the open**, from `config.yaml`, because they are part of your study rather than of this session — the temperature that produced the number in your report is a number your report has to state, and `PLAN.md` asks for it.

`temperature: 0` tells the model to take its most likely answer every time, which is what makes a run repeatable. Raising it is a real experiment rather than a mistake: run the same prompt twice at 0 and twice at 1, and count how many labels changed. That measures how much of the difference between two of your rounds was your prompt and how much was the model — worth knowing before you claim a two-point gain.

Safe to run more than once. With the same settings it hands back the connection it already made; **change one and it reconnects and says so**, rather than quietly going on sending at the old temperature.

In [ ]:
setup(temperature=TEMPERATURE, seed=SEED, model=MODEL)

> **Check the backend line it just printed.** You want:
>
> ```
> LLM backend: Gemini API (gemini-3.1-flash-lite, temperature=0, seed=42)
> ```
>
> If it says **Colab Gemini** instead, no API key was found — put yours in the Colab Secrets panel (the 🔑 icon in the left sidebar) as `GEMINI_API_KEY` and re-run. The keyless backend has no temperature or seed, so the same prompt can give different answers and your numbers will not be reproducible. It must not be your final run.

> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## Step 1 — Open the three files this notebook works from

Now we load the dev half (what you may look at), the pool (where few-shot examples come from) and the **full** gold set.

That third one looks redundant and is not. `build_fewshot` excludes your gold items from the examples it picks — by *text*, since sampling renumbered the ids. Hand it only `dev` and it can pick a **test** item as a worked example, which puts the answer to a held-out item straight into the prompt that produces your headline number. So the full gold set is loaded as an exclusion list, and scored against never.

`LABELS` is read off the full gold set for the same reason: a label that happens to be thin in dev should not quietly shrink your label list.

In [ ]:
# ══ STEP 1 · Load your dev set and your pool ══════════════════════════════
# Opens the three files this notebook works from, and reads your label list
# off the full gold set.
# Creates: dev, pool, gold, LABELS

dev  = load_gold(DEV_PATH)      # what you iterate against, from notebook 03
pool = load_gold(POOL_PATH)     # the spares few-shot examples are drawn from
gold = load_gold(GOLD_PATH)     # ALL of it — as an EXCLUSION list, see above
LABELS = label_set(gold)        # gold, not pool: what you actually adjudicated

print(len(dev), "dev ·", len(pool), "pool ·", LABELS)


## What you have to work with

Every round below is these calls, in an order you choose. Nothing else is imported, and nothing here is new — the right-hand column says where you first ran each one.

| Call | What it gives you | First run |
|---|---|---|
| `load_prompt(path)` | a prompt file, as text | 04 |
| `save_prompt(text, path)` | one version of your prompt, written to a file | 04 |
| `build_fewshot(prompt, pool, gold)` | the same prompt with worked examples in front of it | Day 3, typed by hand |
| `run_prompt(prompt, dev)` | one predicted label per item | Day 3 |
| `evaluate(gold, pred)` | per-class P/R/F1, κ, the matrix, and the macro-F1 back | Day 2 S6 Part B |
| `show_errors(gold, pred)` | just the rows it got wrong | Day 3 |
| `extract_label(reply, labels)` | one label out of one reply | Day 3, as `_extract_level` |
| `last_replies()` | what the model actually said, one string per item | 04 |

### What you can put in a prompt

A prompt is ordinary text with **placeholders** in it. `run_prompt` fills them in, once per item, just before it sends:

| Placeholder | What gets slotted in | Available on |
|---|---|---|
| `{text}` | the sentence being classified | every track |
| `{context}` | the passage it came from | `cars50`, `raamove` |

> **A placeholder is not an f-string.** On Day 1 you wrote `f"... {sentence}"`, and the braces were filled in **on that line**. A prompt template has **no `f`**: the braces stay as they are until `run_prompt` fills them, once per item. Write `PROMPT = f"...{text}"` here and you either get a `NameError` or send the same frozen sentence to every item.

> **Any other `{...}` is an error.** The filling is done with `.format()`, so a stray brace — a JSON example, a `{label}` you meant literally — stops the run on the first item. Write `{{` and `}}` for a literal brace.

`{context}` is the cheapest experiment in the project on the two tracks that have it, and it is already written: `prompts/<track>_context.txt` is the same prompt with the passage added, one `load_prompt` away.

**The number `evaluate` hands back is macro-F1**, and here that is all it is for: something to compare round 2 against round 1 with. Use it to decide what to change next; the number you report comes out of `06_report.ipynb`, where you write the scoring call yourself. If the headline you settled on in `PLAN.md` §9 is a different average, say so beside the rounds table — a dev trail measured one way and a held-out row measured another are not the same ruler.

**And the prompt itself is the other half of what you assemble.** S7 called these the moves available to you; the deck's own split between what to run this week and what to know about is worth re-reading before you start:

| Move | What it changes |
|---|---|
| **Instruction** | what you ask for. Start with a verb, name the label set |
| **Context** | the background or rubric that scopes the task |
| **Input data** | the sentence — and on `cars50` / `raamove`, whether the model also sees the passage it came from |
| **Output indicator** | the *shape* of the answer. This is what `??` rows are about |
| **Persona** | who you tell it to be |
| **Few-shot** | worked examples in the prompt |
| **Chain-of-thought** | asking it to reason before answering |
| **Structured output** | asking for JSON or a fixed form |

*(Know about, do not run this week: self-consistency, RAG, agentic multi-step — S7 says why.)*

**Which of these you try, and in what order, is your experiment.** Few-shot is not the default and not the recommendation; it is one row of that table. `PLAN.md` §8 asks which move you predicted would help, and why.

**`extract_label`** is what turns a reply into a label, and it is where every `??` comes from. Two rules to read: it keeps the **longest** label whose name appears anywhere in the reply — so a reply mentioning two labels is settled by length, not by which one the model meant — and on `Move N` tracks a bare digit anywhere in the reply becomes that move. When your model keeps answering in a shape this cannot read, **this is the cell you change**, and the next one is how your version reaches the loop.

In [ ]:
def extract_label(reply: str, labels: list[str]) -> str:
    """Figure out which of the known labels the model's reply is pointing at.

    Args:
        reply: whatever the model replied.
        labels: the labels your scheme allows.

    Returns:
        The label it found - the longest one when several appear - or "??" when the
        reply contains none of them.

    Example:
        >>> extract_label("I would say B2.", LEVELS)
    """
    reply_text = str(reply).strip()
    reply_lowercased = reply_text.lower()

    # Step 1: collect every known label whose name appears in the reply.
    labels_found = []
    for label in labels:
        if label.lower() in reply_lowercased:
            labels_found.append(label)

    # Step 2: if we found one or more, keep the longest (most specific) one.
    if len(labels_found) > 0:
        longest_label = labels_found[0]
        for label in labels_found:
            if len(label) > len(longest_label):
                longest_label = label
        return longest_label

    # Step 3: special case for "Move 1/2/3" labels - look for a bare digit.
    has_move_labels = False
    for label in labels:
        if label.lower().startswith("move "):
            has_move_labels = True
    if has_move_labels:
        match = re.search(r"\b([1-9])\b", reply_text)
        if match is not None:
            candidate = "Move " + match.group(1)
            if candidate in labels:
                return candidate

    # Step 4: nothing matched.
    return "??"

**`run_prompt`** is the loop: one API call per item, the reply passed through `extract`. Two things worth finding in it. The `prompt.format(...)` line is where `{text}` and `{context}` are filled in, once per item. And `extract` defaults to `extract_label` but is an **argument** — so an edit to the cell above only reaches this loop if you pass it in, which is what the rounds below do. The pacing and retrying happen inside the backend `setup()` opened; that part is plumbing and stays imported.

It keeps the raw replies in `_LAST_REPLIES` rather than returning them, so that `predictions = run_prompt(prompt, dev)` stays the one-line call Day 3 taught. The next cell is how you read them back.

In [ ]:
def run_prompt(prompt: str,
               gold: list[dict[str, str]],
               labels: list[str] | None = None,
               generate_text=None,
               extract=None) -> list[str]:
    """Ask the model to label every item, and collect the predicted labels.

    Same call as Day 3: run_prompt(PROMPT, gold). The optional arguments are worked out
    for you, so you only pass one if you want something different.

    It prints one line per item while it runs - the gold label, the label it read out of
    the reply, and the beginning of the reply itself. Watch the third column when the
    second one says "??": that is the model answering in a shape `extract_label` cannot
    read, which is a finding about your prompt rather than a bug.

    `extract` is how you act on that. `extract_label` decides that "This looks like Move
    2 to me" means `Move 2`, and if your model keeps answering in some shape it misses,
    copy it into a cell, change it, and pass your version in. Pass it rather than only
    redefining it: when this function is the one imported from a file, a redefinition in
    your notebook does not reach the copy this loop calls, and you would get the old
    labels back with no sign anything had been ignored. Passing it also puts the rule
    that produced your numbers in the call, where a reader of the notebook can see it.

    Args:
        prompt: your prompt, containing {text} where the sentence should go, and
            {context} for its passage on the tracks that carry one.
        gold: the items to label.
        labels: the labels your scheme allows. Left out, they are read off `gold`.
        generate_text: the function that sends a prompt. Left out, the backend
            connected by the Setup cell is used.
        extract: the function that turns one reply into one label. Left out,
            `extract_label` is used.

    Returns:
        One predicted label per gold item, in the same order. Replies no label could be
        read out of come back as "??". The raw replies are kept as well - see
        `last_replies()`.

    Example:
        >>> predictions = run_prompt(prompt, dev)
    """
    global _LAST_REPLIES
    if labels is None:
        labels = label_set(gold)
    if generate_text is None:
        generate_text = _default_backend()
    if extract is None:
        extract = extract_label

    # A prompt that asks for {context} on a track whose items have none would quietly
    # send the model an empty passage, once per item, and report a number as if it had
    # tested something. Say so instead.
    if "{context}" in prompt and not any(item.get("context") for item in gold):
        print("WARNING: this prompt uses {context}, but none of these items carry one. "
              "Only the rhetorical-move tracks (cars50, raamove) do. The model is about "
              "to be shown an empty passage " + str(len(gold)) + " times.")

    predictions = []
    replies = []
    total = len(gold)
    position = 0
    # One line per item is readable for a project-sized set and a wall of text for a
    # whole pool, so past that we fall back to a count every ten.
    show_each = total <= 40

    for item in gold:
        position = position + 1
        # Put this item's sentence into the prompt where {text} is - and its passage
        # where {context} is, on the tracks that carry one. A prompt that does not
        # mention {context} simply ignores it.
        filled_prompt = prompt.format(text=item["text"],
                                      context=item.get("context", ""))
        reply = generate_text(filled_prompt)
        predicted_label = extract(reply, labels)
        predictions.append(predicted_label)
        replies.append(str(reply))

        if show_each:
            print(_one_prediction_line(item, predicted_label, reply))
        elif position % 10 == 0:
            print("  ...", position, "/", total, "done")

    # Keep the raw replies. They are the evidence behind every extraction decision -
    # what you look at when a label comes back "??" - and the reproducibility checklist
    # asks for what the model actually said, not only what we made of it.
    _LAST_REPLIES = replies

    # Count how many replies we could not turn into a valid label.
    number_unparseable = 0
    for label in predictions:
        if label == "??":
            number_unparseable = number_unparseable + 1
    print("Got " + str(len(predictions)) + " predictions ("
          + str(number_unparseable) + " could not be parsed).")
    if number_unparseable:
        print("  The ?? rows are replies no label could be read out of. Read what the")
        if show_each:
            print("  model actually said in the right-hand column above before you")
            print("  change anything.")
        else:
            # No per-item lines were printed for a set this size, so pointing at a
            # column that is not there would send them looking for nothing.
            print("  model actually said - `last_replies()` - before you change")
            print("  anything.")
    return predictions

**`last_replies`** hands back what the model actually *said* in the run just finished, one string per item — the evidence behind every `??`. It is here rather than imported for a reason worth knowing: it has to read the `_LAST_REPLIES` that the `run_prompt` **above** writes to. Imported, it would read the one inside `pipeline.py`, which your notebook's loop never touches, and it would hand you an empty list after every run.

In [ ]:
def last_replies() -> list[str]:
    """What the model actually said in the most recent run, one string per item.

    `run_prompt` hands back the labels it read out of these. When one of them is "??",
    or when a label looks wrong, this is where you find out why - the reply is the
    evidence and the label is only our reading of it.

    Returns:
        The raw replies, in the order the items were sent. Empty before the first run.

    Example:
        >>> last_replies()[3]
    """
    return _LAST_REPLIES

**`show_errors`** is the table that decides your next round. Its rule is one line — `item["label"] != predicted` — so a `??` and a confidently wrong label are the same kind of error here, and on a scale a near miss counts the same as a far one. If that is not the view you want, this is the cell.

In [ ]:
def show_errors(gold: list[dict[str, str]],
                predictions: list[str]) -> pd.DataFrame:
    """The items the model got wrong, as a table you can read and argue about.

    F1 tells you whether a round helped. Only this tells you what to change next.

    Args:
        gold: the gold items, each with "id", "text" and "label".
        predictions: one predicted label per gold item, in the same order.

    Returns:
        A table with one row per mistake: id, gold, pred, text. The columns are named
        even when there are no mistakes.

    Example:
        >>> errors = show_errors(gold, predictions)
    """
    rows = []
    for index in range(len(gold)):
        item = gold[index]
        predicted = predictions[index]
        if item["label"] != predicted:
            rows.append({"id": item["id"],
                         "gold": item["label"],
                         "pred": predicted,
                         "text": item["text"]})

    print(len(rows), "of", len(gold), "wrong.")
    # Name the columns even when there are no rows. A table built from an empty list
    # has no columns at all, and then errors["gold"] in the report notebook fails for
    # the one group whose model got everything right - the least deserving group to
    # break on.
    return pd.DataFrame(rows, columns=["id", "gold", "pred", "text"])

## Step 2 — The baseline (round 0)

Your first score, before you have changed anything. Write the plainest prompt that states the task and the label set, run it, score it. **Resist the urge to make it good** — later rounds need something to be measured against, and a baseline you already tuned tells you nothing about whether tuning helped.

**You write the prompt here, in the cell.** It is an ordinary string, so editing it and running the cell again is all it takes to change what gets sent — there is no file to keep in step with it while you are working.

It must contain `{text}`. Everything else is yours.

`f1_by_round` collects one score per round and `NOTES` collects your one-line reason for each. Both are saved at the end of this notebook and printed side by side as your report's prompt-iterations section, so the keys are what your reader sees — name them for what you **changed**, not which round it was.

Now we start those two tables and write the baseline prompt.

In [ ]:
# ══ STEP 2 · Baseline prompt (round 0) ════════════════════════════════════
# Starts the two tables this notebook fills in, and holds the plainest prompt
# you can write for your track.
# Creates: f1_by_round, NOTES, PROMPT

# ✏️ The wording of the baseline. Plainest thing that states the task and
#    names the labels — save the good ideas for the rounds below.

f1_by_round = {}     # round name -> macro-F1 on dev
NOTES = {}           # round name -> why you made that change

# ✏️ Edit this. Keep {text}; it is where each sentence is slotted in.
PROMPT = """Classify the sentence below.
Answer with the label only.

Sentence: {text}"""

# … or start from the file written for your track:
# PROMPT = load_prompt(PROMPT_FILE)

print(PROMPT)


### Now save it as a file

`05_test.ipynb` is a different notebook, and nothing survives between notebooks except what is on disk. A prompt that only ever existed as a string in this session is one you cannot test and cannot report.

Give every version its own name. `v0`, `v1`, `v2` beside each other are what let you show the reader what changed between rounds — and let you go back to the one that scored best after round 3 turned out worse.

Now we write this version to `prompts/`.

In [ ]:
save_prompt(PROMPT, ROOT / "prompts" / (TRACK + "_v0.txt"))

### Now send it to the model

This is the slow cell: one API call per dev item, paced a few seconds apart to stay inside the free tier. It prints one line per item as it goes — the gold label, the label it read out of the reply, and the beginning of the reply itself.

**Watch the third column when the second says `??`.** That is the model answering in a shape `extract_label` cannot read, which is a finding about your prompt — an output-indicator problem — rather than a bug. `last_replies()` gives you the full replies when the printed line is too short to tell.

Note `extract=extract_label`. It names the cell you read above, so the label rule this run used is one you can point at. Written out rather than left to the default, because the default is the copy inside `pipeline.py` — and a reader of your notebook cannot tell which one produced the numbers unless the call says so.

It is on its own **deliberately**. Scoring and reading the errors are separate cells below, so that looking at your results again costs you nothing. If they shared a cell with this one, every re-read would re-run every call and spend your group's quota a second time.

In [ ]:
pred0 = run_prompt(PROMPT, dev, extract=extract_label)

### Now score it

`evaluate` prints the table and the matrix, and hands back the macro-F1, which we store under a name we choose.

`ordered=False` is the safe default. Change it to `True` if — and only if — your labels sit on a **scale** (A1 < A2 < … < C2, Low < Mid < High), where a near miss is a smaller error than a far one. Move 1 / Move 2 / Move 3 are *not* a scale: they are three different jobs, not three amounts.

In [ ]:
f1_by_round["round0 baseline (dev)"] = evaluate(dev, pred0,
                                               ordered=False,
                                               labels=LABELS_ORDER)
NOTES["round0 baseline (dev)"] = "the plainest prompt that states the task"

### Now read what it got wrong

Do not skip this. The errors are the only thing that tells you *what to change*; F1 only tells you afterwards whether the change worked. This is the cell that decides your next round.

In [ ]:
show_errors(dev, pred0)

## Step 3 — Iterate

The loop is always the same, and the middle step is the one that matters:

```
run  →  score  →  READ THE ERRORS  →  change ONE thing  →  run again
```

Before you touch the prompt, look at the error table from the round you just ran and ask **what these misses have in common**. There are only a few answers, and each points at a different row of the moves table above:

| What you see in the errors | Which move it points at |
|---|---|
| One class swallows everything | **instruction** — define that class's boundary, or **few-shot** — show one |
| Two labels traded in both directions | the *distinction* is unclear, to the model and possibly to your coders too |
| Lots of `??` | **output indicator** — it is not answering in the shape you asked for. Fix the instruction, not the definitions |
| Errors scattered with no pattern | you may be at the ceiling of what a prompt can do; consider whether the items are simply hard |

**Three round blocks follow, and they are deliberately identical.** Fill in as many as you use. Each one asks you to say what you expect *before* you run it — that is the Day 3 Part B habit, and it is what makes a round a finding rather than a thing that happened. A prediction you got wrong is worth more than one you never wrote down.

A round that made things **worse** is a result, not a mistake. Keep it in the table. It is often the most informative row you have.

---

### Round 1 — say what you expect, then change one thing

Fill in the three lines, then build the prompt. Leave the block empty if you stop before round 1.

In [ ]:
# ══ STEP 3 · Round 1 — the prediction ═════════════════════════════════════
# Writes down what you are about to change and what you expect it to do,
# before you find out.
# Creates: WORST_1, CHANGE_1, EXPECT_1

# ✏️ Which move from the table, and why that one for these errors.

WORST_1  = "…"     # the class with the lowest F1 last round
CHANGE_1 = "…"     # the ONE move you are making, in a phrase
EXPECT_1 = "…"     # what you expect it to do, and why

print("targeting", WORST_1, "·", CHANGE_1)
print("expecting:", EXPECT_1)


Now write the prompt for this round. **Start from the version you are keeping** — that is `PROMPT` for round 1, and whichever of your later versions scored best after that. Change the one thing you named above and nothing else, or you will not know which change moved the number.

`build_fewshot` is one option among the moves in the table, not the recommendation. It draws examples from the pool while skipping anything in your gold set, so you are not testing the model on answers you just showed it. Note that calling it twice with the same arguments gives you the same prompt both times — the examples are drawn with a fixed seed, so a round that only re-runs it is a round that changes nothing.

In [ ]:
# ✏️ Write this round's prompt. Keep {text}.
PROMPT_v1 = """…"""

# … or few-shot, from the version you are keeping. Note the NEW NAME on
# the left: PROMPT = build_fewshot(PROMPT, ...) would stack examples on
# examples every time you re-ran the cell, silently.
#
# `gold`, not `dev`: the list of items NOT to use as examples has to
# include the test half, or the answers leak into the prompt.
# PROMPT_v1 = build_fewshot(PROMPT, pool, gold,
#                          shots_per_class=1, seed=SEED)

print(PROMPT_v1)

Now save this version, so `05_test.ipynb` can load whichever one you end up choosing.

In [ ]:
save_prompt(PROMPT_v1, ROOT / "prompts" / (TRACK + "_v1.txt"))

Now run it. One API call per dev item, so a re-run costs your group real quota.

In [ ]:
pred1 = run_prompt(PROMPT_v1, dev, extract=extract_label)

Now score it and read the new errors. **Name the key for what you changed** — in the report, *"v1 one example per label"* is an argument and *"round 1"* is a row number. Keep `ordered` the same as the baseline, or the rounds are not comparable.

In [ ]:
KEY_1 = "v1 " + CHANGE_1 + " (dev)"
f1_by_round[KEY_1] = evaluate(dev, pred1, ordered=False,
                            labels=LABELS_ORDER)
NOTES[KEY_1] = EXPECT_1  + "  |  what happened: …"

Now the errors again. Did the ones you were targeting move? Did the confusion matrix change **shape**, or did every cell shift a little? Those two call for different next moves — and whichever it was goes into the `what happened` half of `NOTES[KEY_1]` above.

In [ ]:
show_errors(dev, pred1)

---

### Round 2 — say what you expect, then change one thing

Fill in the three lines, then build the prompt. Leave the block empty if you stop before round 2.

In [ ]:
# ══ STEP 4 · Round 2 — the prediction ═════════════════════════════════════
# Writes down what you are about to change and what you expect it to do,
# before you find out.
# Creates: WORST_2, CHANGE_2, EXPECT_2

# ✏️ Which move from the table, and why that one for these errors.

WORST_2  = "…"     # the class with the lowest F1 last round
CHANGE_2 = "…"     # the ONE move you are making, in a phrase
EXPECT_2 = "…"     # what you expect it to do, and why

print("targeting", WORST_2, "·", CHANGE_2)
print("expecting:", EXPECT_2)


Now write the prompt for this round. **Start from the version you are keeping** — that is `PROMPT` for round 1, and whichever of your later versions scored best after that. Change the one thing you named above and nothing else, or you will not know which change moved the number.

`build_fewshot` is one option among the moves in the table, not the recommendation. It draws examples from the pool while skipping anything in your gold set, so you are not testing the model on answers you just showed it. Note that calling it twice with the same arguments gives you the same prompt both times — the examples are drawn with a fixed seed, so a round that only re-runs it is a round that changes nothing.

In [ ]:
# ✏️ Write this round's prompt. Keep {text}.
PROMPT_v2 = """…"""

# … or few-shot, from the version you are keeping. Note the NEW NAME on
# the left: PROMPT = build_fewshot(PROMPT, ...) would stack examples on
# examples every time you re-ran the cell, silently.
#
# `gold`, not `dev`: the list of items NOT to use as examples has to
# include the test half, or the answers leak into the prompt.
# PROMPT_v2 = build_fewshot(PROMPT, pool, gold,
#                          shots_per_class=1, seed=SEED)

print(PROMPT_v2)

Now save this version, so `05_test.ipynb` can load whichever one you end up choosing.

In [ ]:
save_prompt(PROMPT_v2, ROOT / "prompts" / (TRACK + "_v2.txt"))

Now run it. One API call per dev item, so a re-run costs your group real quota.

In [ ]:
pred2 = run_prompt(PROMPT_v2, dev, extract=extract_label)

Now score it and read the new errors. **Name the key for what you changed** — in the report, *"v2 one example per label"* is an argument and *"round 2"* is a row number. Keep `ordered` the same as the baseline, or the rounds are not comparable.

In [ ]:
KEY_2 = "v2 " + CHANGE_2 + " (dev)"
f1_by_round[KEY_2] = evaluate(dev, pred2, ordered=False,
                            labels=LABELS_ORDER)
NOTES[KEY_2] = EXPECT_2  + "  |  what happened: …"

Now the errors again. Did the ones you were targeting move? Did the confusion matrix change **shape**, or did every cell shift a little? Those two call for different next moves — and whichever it was goes into the `what happened` half of `NOTES[KEY_2]` above.

In [ ]:
show_errors(dev, pred2)

---

### Round 3 — say what you expect, then change one thing

Fill in the three lines, then build the prompt. Leave the block empty if you stop before round 3.

In [ ]:
# ══ STEP 5 · Round 3 — the prediction ═════════════════════════════════════
# Writes down what you are about to change and what you expect it to do,
# before you find out.
# Creates: WORST_3, CHANGE_3, EXPECT_3

# ✏️ Which move from the table, and why that one for these errors.

WORST_3  = "…"     # the class with the lowest F1 last round
CHANGE_3 = "…"     # the ONE move you are making, in a phrase
EXPECT_3 = "…"     # what you expect it to do, and why

print("targeting", WORST_3, "·", CHANGE_3)
print("expecting:", EXPECT_3)


Now write the prompt for this round. **Start from the version you are keeping** — that is `PROMPT` for round 1, and whichever of your later versions scored best after that. Change the one thing you named above and nothing else, or you will not know which change moved the number.

`build_fewshot` is one option among the moves in the table, not the recommendation. It draws examples from the pool while skipping anything in your gold set, so you are not testing the model on answers you just showed it. Note that calling it twice with the same arguments gives you the same prompt both times — the examples are drawn with a fixed seed, so a round that only re-runs it is a round that changes nothing.

In [ ]:
# ✏️ Write this round's prompt. Keep {text}.
PROMPT_v3 = """…"""

# … or few-shot, from the version you are keeping. Note the NEW NAME on
# the left: PROMPT = build_fewshot(PROMPT, ...) would stack examples on
# examples every time you re-ran the cell, silently.
#
# `gold`, not `dev`: the list of items NOT to use as examples has to
# include the test half, or the answers leak into the prompt.
# PROMPT_v3 = build_fewshot(PROMPT, pool, gold,
#                          shots_per_class=1, seed=SEED)

print(PROMPT_v3)

Now save this version, so `05_test.ipynb` can load whichever one you end up choosing.

In [ ]:
save_prompt(PROMPT_v3, ROOT / "prompts" / (TRACK + "_v3.txt"))

Now run it. One API call per dev item, so a re-run costs your group real quota.

In [ ]:
pred3 = run_prompt(PROMPT_v3, dev, extract=extract_label)

Now score it and read the new errors. **Name the key for what you changed** — in the report, *"v3 one example per label"* is an argument and *"round 3"* is a row number. Keep `ordered` the same as the baseline, or the rounds are not comparable.

In [ ]:
KEY_3 = "v3 " + CHANGE_3 + " (dev)"
f1_by_round[KEY_3] = evaluate(dev, pred3, ordered=False,
                            labels=LABELS_ORDER)
NOTES[KEY_3] = EXPECT_3  + "  |  what happened: …"

Now the errors again. Did the ones you were targeting move? Did the confusion matrix change **shape**, or did every cell shift a little? Those two call for different next moves — and whichever it was goes into the `what happened` half of `NOTES[KEY_3]` above.

In [ ]:
show_errors(dev, pred3)

---

## Step 6 — Save the trail, and stop

Both tables go to disk. `05_test.ipynb` reads them back, adds the held-out row, and `06_report.ipynb` prints the two side by side as your report's prompt-iterations section.

**Before you save, fill in the `what happened` half of every `NOTES` entry.** You wrote the prediction before the round; this is where you say whether it held. A row that says only what you expected is half a finding.

Then decide which prompt won on dev, and check that it is **saved as a file** — the next notebook can only load files.

First, look at what you are about to save. A round whose key says only `round 2` is a row number; a round whose key names the move you made is an argument. This is the last easy moment to rename one.

In [ ]:
# ══ STEP 6 · Read the trail back ══════════════════════════════════════════
# Shows the rounds you ran, in order, with the score each one got.
# Nothing new is named — this is a check before you save.

f1_by_round


Now write both tables to disk. `05_test.ipynb` reads them back and adds the held-out row; `06_report.ipynb` prints them side by side.

They overwrite, because re-running this notebook re-runs every round in it — the file has to match the rounds you actually just ran, not a mixture of two sessions.

In [ ]:
save_json(f1_by_round, ROUNDS_PATH, what="rounds", overwrite=True)
save_json(NOTES, NOTES_PATH, what="round notes", overwrite=True)

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> Our baseline scored ___ on dev.
>
> We changed ___ because ___, and expected ___; what happened was ___.
>
> The round that helped most was ___, and we think it worked because ___.
>
> One change we expected to help and it did not was ___.

Write the middle two sentences once per round. The last one is worth as many marks as the rest: a prediction that failed, with a reading of why, is a finding about the model. Dropping the rounds that did not work and reporting only the winner leaves you with a number and nothing to say about it.

---

## 🛑 Before you open `05_test.ipynb`

That notebook opens the held-out set. Once it has been scored, the number is the number — so settle these first:

- Every prompt you might test is **saved as a file** in `prompts/`.
- `PLAN.md` §8 says **which** of them you will test, and **how many**, and how you will pick the winner if you test more than one. Written down before you run, that is a design. Decided afterwards, it is choosing the best of several tries and reporting it as if it were one.
- Every `NOTES` entry says what you expected *and* what happened.

**Next:** `05_test.ipynb`.